<a href="https://colab.research.google.com/github/AngeloSorte/NASA-Planetary-Surface-AI-Classifier-Craters-Dunes-Rocks-Dust-/blob/angelosorte.github.io/%F0%9F%AA%90_NASA_Planetary_Surface_Image_Classifier_Craters_Dunes_Rocks_Dust.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# 🪐 NASA Planetary Surface Image Classifier
# Craters / Dunes / Rocks / Dust
# Dataset: NASA Images API (real data)
# Model: MobileNetV2 Transfer Learning

# =========================
# 1. INSTALL DEPENDENCIES
# =========================
import os
import json
import random
import requests
import numpy as np
from tqdm import tqdm
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from PIL import Image
from io import BytesIO

print("Setup completed 🚀")

# =========================
# 2. DATASET CONFIGURATION
# =========================
BASE_DIR = "/content/nasa_planetary_dataset"

CLASSES = {
    "crater": "mars crater hirise",
    "dune": "mars dune sand",
    "rock": "mars rock surface",
    "dust": "mars dust surface"
}

IMAGES_PER_CLASS = 40
IMAGE_SIZE = (224, 224)

os.makedirs(BASE_DIR, exist_ok=True)

# =========================
# 3. NASA IMAGE DOWNLOADER
# =========================
def fetch_nasa_images(query, max_images=40):
    url = "https://images-api.nasa.gov/search"
    params = {
        "q": query,
        "media_type": "image"
    }

    response = requests.get(url, params=params)
    data = response.json()

    items = data.get("collection", {}).get("items", [])
    image_urls = []

    for item in items:
        try:
            links = item["links"]
            for l in links:
                if "href" in l:
                    image_urls.append(l["href"])
        except:
            continue

    return image_urls[:max_images]


def download_image(url):
    try:
        r = requests.get(url, timeout=10)
        img = Image.open(BytesIO(r.content)).convert("RGB")
        img = img.resize(IMAGE_SIZE)
        return img
    except:
        return None

# =========================
# 4. BUILD DATASET
# =========================
print("Downloading NASA dataset...")

for label, query in CLASSES.items():
    class_dir = os.path.join(BASE_DIR, label)
    os.makedirs(class_dir, exist_ok=True)

    print(f"\n📡 Downloading class: {label}")

    urls = fetch_nasa_images(query, IMAGES_PER_CLASS)

    count = 0
    for url in tqdm(urls):
        img = download_image(url)
        if img is not None:
            img.save(os.path.join(class_dir, f"{label}_{count}.jpg"))
            count += 1

print("\nDataset created ✔️")

# =========================
# 5. DATA GENERATORS
# =========================
datagen = ImageDataGenerator(
    validation_split=0.2,
    preprocessing_function=preprocess_input
)

train_gen = datagen.flow_from_directory(
    BASE_DIR,
    target_size=IMAGE_SIZE,
    batch_size=16,
    class_mode="categorical",
    subset="training"
)

val_gen = datagen.flow_from_directory(
    BASE_DIR,
    target_size=IMAGE_SIZE,
    batch_size=16,
    class_mode="categorical",
    subset="validation"
)

num_classes = len(train_gen.class_indices)
print("Classes:", train_gen.class_indices)

# =========================
# 6. MODEL (MOBILENETV2)
# =========================
base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

# =========================
# 7. TRAINING
# =========================
EPOCHS = 8

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS
)

# =========================
# 8. RANDOM TEST PREDICTION
# =========================
def predict_random_sample():
    class_names = list(train_gen.class_indices.keys())
    test_class = random.choice(class_names)

    test_dir = os.path.join(BASE_DIR, test_class)
    img_file = random.choice(os.listdir(test_dir))
    img_path = os.path.join(test_dir, img_file)

    img = Image.open(img_path).resize(IMAGE_SIZE)
    x = np.expand_dims(np.array(img), axis=0)
    x = preprocess_input(x)

    pred = model.predict(x)
    predicted_class = class_names[np.argmax(pred)]

    print("True label:", test_class)
    print("Predicted:", predicted_class)

predict_random_sample()

# =========================
# 9. SAVE MODEL
# =========================
model.save("nasa_planetary_classifier.keras")
print("Model saved ✔️")

Setup completed 🚀

📡 Downloading class: crater


100%|██████████| 40/40 [00:14<00:00,  2.80it/s]



📡 Downloading class: dune


100%|██████████| 40/40 [00:13<00:00,  2.90it/s]



📡 Downloading class: rock


...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,468 (9.24 MB)

 Trainable params: 164,484 (642.52 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

Epoch 1/8
8/8 ━━━━━━━━━━━━━━━━━━━━ 7s 578ms/step - accuracy: 0.6803 - loss: 0.8087 - val_accuracy: 0.8333 - val_loss: 0.4952
Epoch 2/8
8/8 ━━━━━━━━━━━━━━━━━━━━ 3s 361ms/step - accuracy: 0.9672 - loss: 0.1283 - val_accuracy: 0.9333 - val_loss: 0.1149
Epoch 3/8
8/8 ━━━━━━━━━━━━━━━━━━━━ 3s 359ms/step - accuracy: 0.9918 - loss: 0.0414 - val_accuracy: 1.0000 - val_loss: 0.0506
Epoch 4/8
8/8 ━━━━━━━━━━━━━━━━━━━━ 3s 363ms/step - accuracy: 1.0000 - loss: 0.0185 - val_accuracy: 1.0000 - val_loss: 0.0199
Epoch 5/8
8/8 ━━━━━━━━━━━━━━━━━━━━ 3s 438ms/step - accuracy: 1.0000 - loss: 0.0154 - val_accuracy: 1.0000 - val_loss: 0.0063
Epoch 6/8
8/8 ━━━━━━━━━━━━━━━━━━━━ 5s 361ms/step - accuracy: 1.0000 - loss: 0.0090 - val_accuracy: 1.0000 - val_loss: 0.0047
Epoch 7/8
8/8 ━━━━━━━━━━━━━━━━━━━━ 3s 360ms/step - accuracy: 1.0000 - loss: 0.0043 - val_accuracy: 1.0000 - val_loss: 0.0034
Epoch 8/8
8/8 ━━━━━━━━━━━━━━━━━━━━ 3s 448ms/step - accuracy: 1.0000 - loss: 0.0057 - val_accuracy: 1.0000 - val_loss: 0.0024
